# Simple Assistant Agent

In [ ]:
import sys
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_tavily import TavilySearch
from langgraph.prebuilt import create_react_agent
from langchain.schema import HumanMessage
from langgraph.graph import MessagesState
from langchain_core.messages import HumanMessage
from langgraph.graph import START, StateGraph
from langgraph.prebuilt import tools_condition, ToolNode
from IPython.display import Image

load_dotenv()

if not os.environ.get("ANTHROPIC_API_KEY"):
    print("environment variable ANTHROPIC_API_KEY not set")
    sys.exit(1)

if not os.environ.get("TAVILY_API_KEY"):
    print("environment variable TAVILY_API_KEYILY not set")
    sys.exit(1)

### Create the agent and tools

In [ ]:
llm = init_chat_model("claude-3-5-sonnet-latest", model_provider="anthropic")

# Initialise search tool
tavily_search_tool = TavilySearch(
    max_results=5,
    topic="general"
)


def multiply(a: int, b: int) -> int:
    """
    Takes two integers and multiplies them together
    """
    return a * b

agent = create_react_agent(
    llm,
    tools=[tavily_search_tool, multiply],
    prompt="You are a GRC analyst that is the subject matter expert for ISO27001. You must return information that is up-to-date."
    )


tools = [tavily_search_tool, multiply]

### Build the graph

In [ ]:
# Graph
builder = StateGraph(MessagesState)

# Define nodes: these do the work
builder.add_node("assistant", agent)
builder.add_node("tools", ToolNode(tools))

# Define edges: these determine how the control flow moves
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    # If the latest message (result) from assistant is a tool call -> tools_condition routes to tools
    # If the latest message (result) from assistant is a not a tool call -> tools_condition routes to END
    tools_condition,
)
builder.add_edge("tools", "assistant")
react_graph = builder.compile()

# Show
Image(react_graph.get_graph().draw_png())

### Tool calling - Python function

In [ ]:
messages = [HumanMessage(content="What is 200 times 5")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

### Tool calling - web search

In [ ]:
messages = [HumanMessage(content="What are the ISO27001 guidelines for password length and complexity.")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

### LLM decides not to call a tool

In [ ]:
messages = [HumanMessage(content="Hi there")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()

### Call multiple tools

In [ ]:
messages = [HumanMessage(content="Multiply 4 by 5 and then tell me something interesting about ISO27001")]
messages = react_graph.invoke({"messages": messages})
for m in messages['messages']:
    m.pretty_print()